# 🏎️ Motorsport Race Performance and Strategy Management System
## 2023 Season Offline Fast Importer

All official 2023 F1 dataset files have been downloaded locally to `data/2023/`.
**No external API calls or rate-limits** will occur. This entire import runs from local files in ~3 seconds.

### 1. Imports and Local Parsing Utilities

In [15]:
import json
import os
import re
from pathlib import Path
from getpass import getpass
import mysql.connector

# Locate project root whether notebook runs from root or scripts/ folder
current_dir = Path.cwd()
if current_dir.name == "scripts":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data" / "2023"
print(f"Using dataset directory: {DATA_DIR.resolve()}")

def load_local_json(filename):
    path = DATA_DIR / filename
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["MRData"]

def parse_lap_time(time_str):
    """Convert lap time string (e.g. '1:31.295') to MySQL TIME '00:01:31.295'."""
    if not time_str or not isinstance(time_str, str):
        return None
    time_str = time_str.strip()
    if not time_str:
        return None

    m = re.match(r"^(\d+):(\d{2})\.(\d+)$", time_str)
    if m:
        mins, secs, millis = m.groups()
        return f"00:{int(mins):02d}:{int(secs):02d}.{millis[:3].ljust(3, '0')}"

    m2 = re.match(r"^(\d+)\.(\d+)$", time_str)
    if m2:
        secs, millis = m2.groups()
        mins = int(secs) // 60
        rem_secs = int(secs) % 60
        return f"00:{mins:02d}:{rem_secs:02d}.{millis[:3].ljust(3, '0')}"

    return None

def parse_duration(duration_str):
    """Convert duration string (e.g. '25.885' or '15:25.181') to decimal seconds."""
    if not duration_str:
        return None
    try:
        duration_str = str(duration_str).strip()
        if ":" in duration_str:
            parts = duration_str.split(":")
            return round(float(parts[0]) * 60 + float(parts[1]), 3)
        return round(float(duration_str), 3)
    except (ValueError, TypeError):
        return None

def get_next_id(cursor, table_name, pk_name):
    cursor.execute(f"SELECT COALESCE(MAX({pk_name}), 0) + 1 FROM {table_name}")
    return cursor.fetchone()[0]

def build_lookups(cursor):
    cursor.execute("SELECT jolpica_constructor_id, team_id FROM team WHERE jolpica_constructor_id IS NOT NULL")
    team_map = dict(cursor.fetchall())

    cursor.execute("SELECT jolpica_driver_id, driver_id FROM driver WHERE jolpica_driver_id IS NOT NULL")
    driver_map = dict(cursor.fetchall())

    cursor.execute("SELECT jolpica_circuit_id, circuit_id FROM circuit WHERE jolpica_circuit_id IS NOT NULL")
    circuit_map = dict(cursor.fetchall())

    cursor.execute("SELECT year, season_id FROM season")
    season_map = dict(cursor.fetchall())

    cursor.execute("SELECT season_id, round_no, race_id FROM race")
    race_map = {(row[0], row[1]): row[2] for row in cursor.fetchall()}

    return {
        "teams": team_map,
        "drivers": driver_map,
        "circuits": circuit_map,
        "seasons": season_map,
        "races": race_map
    }

print("✓ Local utilities and dataset paths ready.")

Using dataset directory: C:\Users\manav\Documents\manav\PES\sem 5\DBMS\mini-project\data\2023
✓ Local utilities and dataset paths ready.


### 2. Connect to MySQL Database & Fix Duration Column

In [16]:
password = getpass("Enter MySQL root password: ")

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="motorsport_db"
)
cursor = connection.cursor()
print("✓ Connected to motorsport_db.")

# Ensure duration column can hold red-flag pit stops
try:
    cursor.execute("ALTER TABLE pit_stop MODIFY COLUMN duration DECIMAL(8,3);")
    connection.commit()
    print("✓ pit_stop.duration verified as DECIMAL(8,3).")
except Exception as e:
    print(f"Note on ALTER TABLE: {e}")

✓ Connected to motorsport_db.
✓ pit_stop.duration verified as DECIMAL(8,3).


### 3. Step 1: Import Reference Data (Season, Teams, Drivers, Circuits)

In [17]:
# 1. Season
races_raw = load_local_json("races.json")["RaceTable"]["Races"]
dates = [r["date"] for r in races_raw]
start_date = min(dates) if dates else "2023-03-01"
end_date = max(dates) if dates else "2023-11-30"

cursor.execute("SELECT season_id FROM season WHERE year = 2023")
row = cursor.fetchone()
if row:
    cursor.execute("UPDATE season SET start_date = %s, end_date = %s WHERE season_id = %s", (start_date, end_date, row[0]))
else:
    next_season_id = get_next_id(cursor, "season", "season_id")
    cursor.execute(
        "INSERT INTO season (season_id, year, start_date, end_date) VALUES (%s, %s, %s, %s)",
        (next_season_id, 2023, start_date, end_date)
    )
connection.commit()
print("✓ Season 2023 imported.")

# 2. Teams
teams_raw = load_local_json("constructors.json")["ConstructorTable"]["Constructors"]
for t in teams_raw:
    cursor.execute("SELECT team_id FROM team WHERE jolpica_constructor_id = %s", (t["constructorId"],))
    existing = cursor.fetchone()
    if existing:
        cursor.execute(
            "UPDATE team SET team_name = %s, nationality = %s WHERE team_id = %s",
            (t["name"], t.get("nationality"), existing[0])
        )
    else:
        next_team_id = get_next_id(cursor, "team", "team_id")
        cursor.execute(
            "INSERT INTO team (team_id, team_name, nationality, base, jolpica_constructor_id) VALUES (%s, %s, %s, %s, %s)",
            (next_team_id, t["name"], t.get("nationality"), None, t["constructorId"])
        )
connection.commit()
print(f"✓ {len(teams_raw)} Teams imported.")

# 3. Drivers
drivers_raw = load_local_json("drivers.json")["DriverTable"]["Drivers"]
for d in drivers_raw:
    driver_name = f"{d['givenName']} {d['familyName']}"
    cursor.execute("SELECT driver_id FROM driver WHERE jolpica_driver_id = %s", (d["driverId"],))
    existing = cursor.fetchone()
    if existing:
        cursor.execute(
            "UPDATE driver SET driver_name = %s, nationality = %s, dob = %s WHERE driver_id = %s",
            (driver_name, d.get("nationality"), d.get("dateOfBirth"), existing[0])
        )
    else:
        next_driver_id = get_next_id(cursor, "driver", "driver_id")
        cursor.execute(
            "INSERT INTO driver (driver_id, driver_name, nationality, dob, team_id, jolpica_driver_id) VALUES (%s, %s, %s, %s, %s, %s)",
            (next_driver_id, driver_name, d.get("nationality"), d.get("dateOfBirth"), None, d["driverId"])
        )
connection.commit()
print(f"✓ {len(drivers_raw)} Drivers imported.")

# 4. Circuits
for r in races_raw:
    circ = r["Circuit"]
    loc = circ.get("Location", {})
    cursor.execute("SELECT circuit_id FROM circuit WHERE jolpica_circuit_id = %s", (circ["circuitId"],))
    existing = cursor.fetchone()
    if existing:
        cursor.execute(
            "UPDATE circuit SET circuit_name = %s, location = %s, country = %s WHERE circuit_id = %s",
            (circ["circuitName"], loc.get("locality"), loc.get("country"), existing[0])
        )
    else:
        next_circuit_id = get_next_id(cursor, "circuit", "circuit_id")
        cursor.execute(
            "INSERT INTO circuit (circuit_id, circuit_name, location, country, jolpica_circuit_id) VALUES (%s, %s, %s, %s, %s)",
            (next_circuit_id, circ["circuitName"], loc.get("locality"), loc.get("country"), circ["circuitId"])
        )
connection.commit()
print(f"✓ Circuits from {len(races_raw)} races imported.")

✓ Season 2023 imported.
✓ 10 Teams imported.
✓ 22 Drivers imported.
✓ Circuits from 22 races imported.


### 4. Step 2: Import Races

In [18]:
lookups = build_lookups(cursor)
season_id = lookups["seasons"].get(2023)
races_raw = load_local_json("races.json")["RaceTable"]["Races"]

for r in races_raw:
    circuit_id = lookups["circuits"].get(r["Circuit"]["circuitId"])
    round_no = int(r["round"])
    race_date = r["date"]

    cursor.execute("SELECT race_id FROM race WHERE season_id = %s AND round_no = %s", (season_id, round_no))
    existing = cursor.fetchone()
    if existing:
        cursor.execute(
            "UPDATE race SET circuit_id = %s, race_date = %s WHERE race_id = %s",
            (circuit_id, race_date, existing[0])
        )
    else:
        next_race_id = get_next_id(cursor, "race", "race_id")
        cursor.execute(
            "INSERT INTO race (race_id, season_id, circuit_id, round_no, race_date) VALUES (%s, %s, %s, %s, %s)",
            (next_race_id, season_id, circuit_id, round_no, race_date)
        )
connection.commit()
print(f"✓ {len(races_raw)} Races imported.")

✓ 22 Races imported.


### 5. Step 3: Import Race Results

In [19]:
lookups = build_lookups(cursor)
season_id = lookups["seasons"].get(2023)
races_with_results = load_local_json("results.json")["RaceTable"]["Races"]

total_results = 0
for race_data in races_with_results:
    round_no = int(race_data["round"])
    race_id = lookups["races"].get((season_id, round_no))
    if not race_id:
        continue

    results = race_data.get("Results", [])
    for res in results:
        driver_id = lookups["drivers"].get(res["Driver"]["driverId"])
        team_id = lookups["teams"].get(res["Constructor"]["constructorId"])

        if driver_id and team_id:
            cursor.execute(
                "UPDATE driver SET team_id = %s WHERE driver_id = %s AND team_id IS NULL",
                (team_id, driver_id)
            )

        pos = int(res["position"]) if res.get("position", "").isdigit() else None
        grid = int(res["grid"]) if res.get("grid", "").isdigit() else None
        laps = int(res["laps"]) if res.get("laps", "").isdigit() else 0
        points = float(res.get("points", 0.0))
        status = res.get("status", "Unknown")

        cursor.execute("SELECT result_id FROM race_result WHERE race_id = %s AND driver_id = %s", (race_id, driver_id))
        existing = cursor.fetchone()
        if existing:
            cursor.execute(
                """
                UPDATE race_result
                SET position = %s, grid_position = %s, laps_completed = %s, points = %s, status = %s, team_id = %s
                WHERE result_id = %s
                """,
                (pos, grid, laps, points, status, team_id, existing[0])
            )
        else:
            next_res_id = get_next_id(cursor, "race_result", "result_id")
            cursor.execute(
                """
                INSERT INTO race_result (result_id, race_id, driver_id, position, grid_position, laps_completed, points, status, team_id)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (next_res_id, race_id, driver_id, pos, grid, laps, points, status, team_id)
            )
        total_results += 1

connection.commit()
print(f"✓ Total {total_results} race results imported.")

✓ Total 100 race results imported.


### 6. Step 4: Import Qualifying Results

In [20]:
lookups = build_lookups(cursor)
season_id = lookups["seasons"].get(2023)
races_with_q = load_local_json("qualifying.json")["RaceTable"]["Races"]

total_qualifying = 0
for race_data in races_with_q:
    round_no = int(race_data["round"])
    race_id = lookups["races"].get((season_id, round_no))
    if not race_id:
        continue

    q_results = race_data.get("QualifyingResults", [])
    for q in q_results:
        driver_id = lookups["drivers"].get(q["Driver"]["driverId"])
        pos = int(q["position"]) if q.get("position", "").isdigit() else None
        q1 = parse_lap_time(q.get("Q1"))
        q2 = parse_lap_time(q.get("Q2"))
        q3 = parse_lap_time(q.get("Q3"))

        cursor.execute("SELECT qualifying_id FROM qualifying_result WHERE race_id = %s AND driver_id = %s", (race_id, driver_id))
        existing = cursor.fetchone()
        if existing:
            cursor.execute(
                "UPDATE qualifying_result SET position = %s, q1_time = %s, q2_time = %s, q3_time = %s WHERE qualifying_id = %s",
                (pos, q1, q2, q3, existing[0])
            )
        else:
            next_qid = get_next_id(cursor, "qualifying_result", "qualifying_id")
            cursor.execute(
                """
                INSERT INTO qualifying_result (qualifying_id, race_id, driver_id, position, q1_time, q2_time, q3_time)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                """,
                (next_qid, race_id, driver_id, pos, q1, q2, q3)
            )
        total_qualifying += 1

connection.commit()
print(f"✓ Total {total_qualifying} qualifying results imported.")

✓ Total 100 qualifying results imported.


### 7. Step 5: Import Pit Stops (All 22 Rounds from Local Files)

In [21]:
lookups = build_lookups(cursor)
season_id = lookups["seasons"].get(2023)

total_stops = 0
for round_no in range(1, 23):
    race_id = lookups["races"].get((season_id, round_no))
    if not race_id:
        continue

    filename = os.path.join("pitstops", f"round_{round_no}.json")
    p_data = load_local_json(filename)["RaceTable"]["Races"]
    if not p_data or not p_data[0].get("PitStops"):
        continue

    stops = p_data[0]["PitStops"]
    for p in stops:
        driver_id = lookups["drivers"].get(p["driverId"])
        lap_number = int(p["lap"])
        stop_time = p.get("time")
        duration = parse_duration(p.get("duration"))

        cursor.execute(
            "SELECT stop_id FROM pit_stop WHERE race_id = %s AND driver_id = %s AND lap_number = %s",
            (race_id, driver_id, lap_number)
        )
        existing = cursor.fetchone()
        if existing:
            cursor.execute(
                "UPDATE pit_stop SET stop_time = %s, duration = %s WHERE stop_id = %s",
                (stop_time, duration, existing[0])
            )
        else:
            next_sid = get_next_id(cursor, "pit_stop", "stop_id")
            cursor.execute(
                """
                INSERT INTO pit_stop (stop_id, race_id, driver_id, lap_number, stop_time, duration)
                VALUES (%s, %s, %s, %s, %s, %s)
                """,
                (next_sid, race_id, driver_id, lap_number, stop_time, duration)
            )
        total_stops += 1

connection.commit()
print(f"✓ Total {total_stops} pit stops imported across all 22 rounds!")

✓ Total 911 pit stops imported across all 22 rounds!


### 8. Step 6: Import Driver Standings

In [22]:
lookups = build_lookups(cursor)
season_id = lookups["seasons"].get(2023)

standings_raw = load_local_json("driverstandings.json")["StandingsTable"]["StandingsLists"]
driver_standings = standings_raw[0]["DriverStandings"]

for s in driver_standings:
    driver_id = lookups["drivers"].get(s["Driver"]["driverId"])
    pos = int(s["position"]) if s.get("position", "").isdigit() else None
    points = float(s.get("points", 0.0))
    wins = int(s.get("wins", 0))

    cursor.execute("SELECT standing_id FROM driver_standing WHERE season_id = %s AND driver_id = %s", (season_id, driver_id))
    existing = cursor.fetchone()
    if existing:
        cursor.execute(
            "UPDATE driver_standing SET position = %s, points = %s, wins = %s WHERE standing_id = %s",
            (pos, points, wins, existing[0])
        )
    else:
        next_sid = get_next_id(cursor, "driver_standing", "standing_id")
        cursor.execute(
            """
            INSERT INTO driver_standing (standing_id, season_id, driver_id, position, points, wins)
            VALUES (%s, %s, %s, %s, %s, %s)
            """,
            (next_sid, season_id, driver_id, pos, points, wins)
        )

connection.commit()
print(f"✓ {len(driver_standings)} Driver standings imported.")

✓ 22 Driver standings imported.


### 9. Final Verification: Row Counts Across All Tables

In [23]:
print("==================================================")
print("Final Summary of Row Counts in motorsport_db:")
tables = ["season", "team", "driver", "circuit", "race", "race_result", "qualifying_result", "pit_stop", "driver_standing"]
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  {table:20s}: {count} rows")
print("==================================================")

# Close connection cleanly
cursor.close()
connection.close()
print("✓ MySQL connection closed cleanly.")

Final Summary of Row Counts in motorsport_db:
  season              : 1 rows
  team                : 10 rows
  driver              : 22 rows
  circuit             : 22 rows
  race                : 22 rows
  race_result         : 440 rows
  qualifying_result   : 440 rows
  pit_stop            : 911 rows
  driver_standing     : 22 rows
✓ MySQL connection closed cleanly.
